# Modelado de Fatiga con xLSTM (Extended Long Short-Term Memory) - Variante sLSTM

Este notebook contiene la explicación teórica, la revisión de literatura científica, la arquitectura detallada y la implementación paso a paso de una red **xLSTM (variante sLSTM - Stabilized LSTM)** en PyTorch (empleando compuertas exponenciales y un mecanismo de estabilización numérica con acumuladores y máximos) para predecir los niveles continuos de fatiga física y mental del dataset **FatigueSet**.

---

## 1. Fundamentos Teóricos y Literatura de Referencia

La arquitectura **xLSTM (Extended Long Short-Term Memory)** fue propuesta en 2024 por Beck et al. como una evolución de la LSTM clásica para competir con las arquitecturas Transformer en el procesamiento de secuencias largas. xLSTM supera las limitaciones de la LSTM mediante dos innovaciones clave:
1. **Compuertas Exponenciales:** Permiten una selección más dinámica y agresiva de la información.
2. **Estructura de Normalización y Estabilización:** Previene el desbordamiento numérico (overflow) de las compuertas exponenciales.

### sLSTM (Scalar LSTM) y Estabilización Numérica

La variante **sLSTM** mantiene una memoria escalar tradicional pero introduce puertas exponenciales. Para evitar el overflow numérico, sLSTM implementa una técnica análoga al cálculo del *safe softmax*, manteniendo dos estados adicionales por cada celda:
1. **Estado del Estabilizador ($m_t$):** Registra el logaritmo del máximo valor pre-activado acumulado.
2. **Normalizador ($n_t$):** Acumula la contribución ponderada de las compuertas exponenciales de entrada.

#### Formulación Matemática del Paso Temporal sLSTM:

Dada la entrada $x_t$, el estado oculto previo $h_{t-1}$ y el estado de la celda previo $c_{t-1}$:

1. **Pre-activaciones lineales:**
   $$\tilde{f}_t = W_f x_t + U_f h_{t-1} + b_f$$
   $$\tilde{i}_t = W_i x_t + U_i h_{t-1} + b_i$$
   $$\tilde{o}_t = W_o x_t + U_o h_{t-1} + b_o$$
   $$\tilde{z}_t = W_z x_t + U_z h_{t-1} + b_z$$

2. **Actualización del Estabilizador Máximo:**
   $$m_t = \max(\tilde{f}_t + m_{t-1}, \tilde{i}_t)$$

3. **Compuertas Estabilizadas:**
   $$f_t = \exp(\tilde{f}_t + m_{prev} - m_t)$$
   $$i_t = \exp(\tilde{i}_t - m_t)$$
   $$o_t = \sigma(\tilde{o}_t)$$

4. **Actualización del Estado de Memoria y Normalizador:**
   $$z_t = \tanh(\tilde{z}_t)$$
   $$c_t = f_t \odot c_{t-1} + i_t \odot z_t$$
   $$n_t = f_t \odot n_{t-1} + i_t$$

5. **Estado Oculto de Salida Normalizado:**
   $$h_t = o_t \odot \frac{c_t}{n_t + \epsilon}$$

---

### Diagrama de Flujo de la Celda sLSTM (Mermaid)

```mermaid
graph TD
    subgraph "Celda sLSTM (Paso t)"
        xt["Entrada: x_t"]
        h_prev["Oculto previo: h_{t-1}"]
        c_prev["Celda previa: c_{t-1}"]
        n_prev["Normalizador previo: n_{t-1}"]
        m_prev["Estabilizador previo: m_{t-1}"]
        f_tilde["f̃_t = W_f x_t + U_f h_{t-1} + b_f"]
        i_tilde["ĩ_t = W_i x_t + U_i h_{t-1} + b_i"]
        z_tilde["z̃_t = W_z x_t + U_z h_{t-1} + b_z"]
        o_tilde["õ_t = W_o x_t + U_o h_{t-1} + b_o"]
        m_update["m_t = max(m_{t-1} + f̃_t, ĩ_t)"]
        f_exp["f_t = exp(f̃_t + m_{t-1} - m_t)"]
        i_exp["i_t = exp(ĩ_t - m_t)"]
        o_sig["o_t = σ(õ_t)"]
        z_tanh["z_t = tanh(z̃_t)"]
        c_update["c_t = f_t ⊙ c_{t-1} + i_t ⊙ z_t"]
        n_update["n_t = f_t ⊙ n_{t-1} + i_t"]
        norm["División: c_t / (n_t + ε)"]
        h_update["h_t = o_t ⊙ (c_t / (n_t + ε))"]
        xt --> f_tilde & i_tilde & z_tilde & o_tilde
        h_prev --> f_tilde & i_tilde & z_tilde & o_tilde
        f_tilde --> m_update
        i_tilde --> m_update
        m_prev --> m_update
        m_update --> f_exp & i_exp
        m_prev --> f_exp
        f_tilde --> f_exp
        i_tilde --> i_exp
        c_prev --> c_update
        f_exp --> c_update & n_update
        i_exp --> c_update & n_update
        z_tanh --> c_update
        n_prev --> n_update
        c_update --> norm
        n_update --> norm
        norm --> h_update
        o_sig --> h_update
    end
```

---

### Referencias Bibliográficas Científicas

* **Beck, M., Pöppel, K., Spanring, M., Gesslhuber, A., Kopp, T., Klambauer, G., Brandstetter, J., & Hochreiter, S. (2024).** *xLSTM: Extended Long Short-Term Memory*. arXiv preprint arXiv:2405.04517. [Enlace al Paper](https://arxiv.org/abs/2405.04517)

In [1]:
# SETUP e IMPORTACIONES
import sys
import time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

# Añadir fatigueset-lib al sys.path
lib_path = str(Path.cwd().parent / "fatigueset-lib")
if lib_path not in sys.path:
    sys.path.insert(0, lib_path)

from fatigueset import FatigueSetPipeline
from fatigueset.models import CustomxLSTMRegressor, FatigueSequenceDataset
from fatigueset.models.rnn import _prepare_target_table, _merge_raw_streams, _build_sequences

print("[OK] Imports completados y path configurado.")
print(f"Versión de PyTorch: {torch.__version__}")
print(f"Dispositivo actual: {'cuda' if torch.cuda.is_available() else 'cpu'}")

[OK] Imports completados y path configurado.
Versión de PyTorch: 2.6.0+cu124
Dispositivo actual: cuda


## 2. Configuración del Pipeline y Construcción de Secuencias

Cargamos los datos fisiológicos del dataset `fatigueset` y alineamos los flujos de sensores de pecho (Chest) y muñeca (Wrist) construyendo tensores secuenciales de 128 timesteps con un paso de 32.

In [2]:
# Configuración del dataset y pipeline
dataset_path = str(Path.cwd().parent / "fatigueset")
pipeline = FatigueSetPipeline(dataset_path=dataset_path, umbral_nulos=5.0)

print("Cargando dataset...")
raw = pipeline.cargar_dataset(verbose=False)

print("Preparando targets del dataframe ML...")
df_ml = pipeline.construir_dataset_ml(raw)
df_targets = _prepare_target_table(df_ml)

print("Combinando streams fisiológicos crudos (Chest y Wrist)....")
df_raw = _merge_raw_streams(raw)

# Parámetros de ventanas de secuencia temporal
seq_len = 128
step = 32

print(f"Construyendo secuencias de tamaño={seq_len} y paso={step}...")
X_arr, y_arr, groups, feature_cols = _build_sequences(
    df_raw=df_raw,
    df_targets=df_targets,
    seq_len=seq_len,
    step=step
)

print(f"[OK] Dimensiones de tensores construidos:")
print(f"  - X: {X_arr.shape} (Número de ventanas x seq_len x features)")
print(f"  - y: {y_arr.shape} (Número de ventanas x 2 targets)")
print(f"  - Columnas de sensores: {len(feature_cols)}")

Cargando dataset...
Preparando targets del dataframe ML...
Combinando streams fisiológicos crudos (Chest y Wrist)....
Construyendo secuencias de tamaño=128 y paso=32...
[OK] Dimensiones de tensores construidos:
  - X: (1306, 128, 23) (Número de ventanas x seq_len x features)
  - y: (1306, 2) (Número de ventanas x 2 targets)
  - Columnas de sensores: 23


## 3. División de Datos por Participante (Group Split)

Para prevenir la fuga de información (data leakage), seleccionamos al participante `'01'` exclusivamente para la validación cruzada y el resto para entrenamiento.

In [3]:
train_idx = np.where(groups != '01')[0]
val_idx = np.where(groups == '01')[0]

X_train, y_train = X_arr[train_idx], y_arr[train_idx]
X_val, y_val = X_arr[val_idx], y_arr[val_idx]

train_dataset = FatigueSequenceDataset(X_train, y_train)
val_dataset = FatigueSequenceDataset(X_val, y_val)

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

print(f"Train samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")

Train samples: 1184
Validation samples: 122


## 4. Inicialización del Regresor xLSTM

Instanciamos nuestro regresor xLSTM (sLSTM) personalizado. Definimos una arquitectura con un tamaño oculto de 64, 2 capas de profundidad, y una probabilidad de dropout de 0.2.

In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"

input_size = len(feature_cols)
hidden_size = 64
num_layers = 2
dropout = 0.2

model = CustomxLSTMRegressor(
    input_size=input_size,
    hidden_size=hidden_size,
    num_layers=num_layers,
    dropout=dropout,
    output_size=2
).to(device)

print(model)

CustomxLSTMRegressor(
  (lstm): CustomsLSTM(
    (layers): ModuleList(
      (0-1): 2 x CustomsLSTMCell()
    )
    (dropout_layer): Dropout(p=0.2, inplace=False)
  )
  (fc): Linear(in_features=64, out_features=2, bias=True)
)


## 5. Entrenamiento de Validación (5 Épocas)

Entrenamos el modelo durante 5 épocas empleando un optimizador Adam, una tasa de aprendizaje de $10^{-3}$ y gradient clipping para mantener la estabilidad del entrenamiento.

In [5]:
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

epochs = 5
print("Iniciando entrenamiento...")

for epoch in range(1, epochs + 1):
    # Modo entrenamiento
    model.train()
    total_train_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        
        optimizer.zero_grad()
        preds = model(xb)
        loss = loss_fn(preds, yb)
        loss.backward()
        
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        
        total_train_loss += loss.item()
        
    avg_train_loss = total_train_loss / len(train_loader)
    
    # Modo validación
    model.eval()
    total_val_loss = 0.0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            preds = model(xb)
            loss = loss_fn(preds, yb)
            total_val_loss += loss.item()

    avg_val_loss = total_val_loss / len(val_loader) if len(val_loader) > 0 else 0.0
    
    print(f"Epoch {epoch}/{epochs} - Train Loss (MSE): {avg_train_loss:.6f} - Val Loss (MSE): {avg_val_loss:.6f}")

print("[OK] Entrenamiento finalizado correctamente.")

Iniciando entrenamiento...
Epoch 1/5 - Train Loss (MSE): 1246.820476 - Val Loss (MSE): 1120.202728
Epoch 2/5 - Train Loss (MSE): 1027.594006 - Val Loss (MSE): 919.342285
Epoch 3/5 - Train Loss (MSE): 886.495066 - Val Loss (MSE): 769.546051
Epoch 4/5 - Train Loss (MSE): 779.197651 - Val Loss (MSE): 641.817047
Epoch 5/5 - Train Loss (MSE): 686.813907 - Val Loss (MSE): 531.164246
[OK] Entrenamiento finalizado correctamente.


## 6. Serialización del Modelo

Guardamos los pesos del modelo en el directorio `/models/deep_learning/`.

In [6]:
output_dir = Path.cwd().parent / "models" / "deep_learning"
output_dir.mkdir(parents=True, exist_ok=True)

model_path = output_dir / "xlstm_fatigue_notebook.pt"
torch.save(model.state_dict(), model_path)

print(f"[OK] Modelo guardado exitosamente en: {model_path}")

[OK] Modelo guardado exitosamente en: c:\Users\egull\OneDrive\Documentos\Proyectos\tfg\models\deep_learning\xlstm_fatigue_notebook.pt
